In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir(r'C:\Users\Lenovo\Desktop\Diabetes_Classifier\notebooks')
os.chdir("..")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [3]:
from modeling.XGBoost import XGBoost
from modeling.RandomForest import RandomForest
from modeling.KNN import KNN
from modeling.MLP import MLP
from modeling.SVC import SVC
from modeling.NB import NB
from modeling.Ensemble import LR
from src.data_feature_engineering import features
from src.data_preprocessing import scaling
import pandas as pd
import numpy as np
from modeling.Ada import Ada
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,recall_score,precision_score,balanced_accuracy_score,roc_auc_score,accuracy_score
import warnings
warnings.filterwarnings('ignore')

c:\Users\Lenovo\anaconda3\envs\Diabetes\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PATH="data/clean/Diabetes Classifier Clean.csv"
df=pd.read_csv(PATH)
df.head()
df.columns = df.columns.str.strip()

In [5]:
df.head()
# df=df.drop('anomaly_score',axis=1)

df = df[df['cr'] != 800]

# Check zero counts per numeric column
for col in df.select_dtypes(include='number').columns:
    zero_pct = (df[col] == 0).mean() * 100
    if 0 < zero_pct < 50:  # some zeros but not the majority (avoid flagging genuinely binary/count cols)
        print(f"{col}: {zero_pct:.2f}% zeros, min={df[col].min()}, max={df[col].max()}")
        
df['gender']=df['gender'].map({'M':1,"F":0})

In [6]:


y=df['diagnosis']
X=df.drop(['diagnosis'],axis=1)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_test=features(X_train,X_test)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)


In [7]:
X_eval=pd.concat([X_train,X_test])
y_eval=pd.concat([y_train,y_test])

In [8]:

xgb=XGBoost()
xgb_params={'n_estimators': 7800, 'max_depth': 13, 'learning_rate': 0.03585979413065538, 'gamma': 0.00025046646096832056, 'min_child_weight': 8, 'reg_alpha': 9.979812055651985, 'reg_lambda': 0.041477402642739976, 'subsample': 0.9999355054935987, 'colsample_bytree': 0.8896668801909641, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)
pd.set_option('display.max_rows',None)
m=xgb.evaluation(X_eval,y_eval)
m.out()
#Optuna
# params=xgb.hyperparameter_tuning(X,y,30)
# xgb.set_params(params)



Accuracy: 0.8223800694147293, std: 0.00822070530809183
Precision: 0.7855624361317548, std: 0.0076426793915217745
F1: 0.7650266076401444, std: 0.012823396520061783
Recall: 0.7457286432160805, std: 0.02006277867103229
Balanced Accuracy:0.8083522887755248, std: 0.010211595603912292
ROC-AUC: 0.9106683622809341, std: 0.006544309935578303


In [9]:
rf=RandomForest()
rf_params={'n_estimators': 1200, 'max_depth': 6, 'min_samples_split': 18, 'min_samples_leaf': 9, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
# rf_params=rf.hyperparameter_tuning(X_train,y_train,30)
rf.set_params(rf_params)
m=rf.evaluation(X_eval,y_eval)
m.out()



Accuracy: 0.8286203584842866, std: 0.013427863453361897
Precision: 0.798714482059327, std: 0.014758062091412703
F1: 0.7714478612579825, std: 0.019727958513363272
Recall: 0.7462311557788945, std: 0.02692079444417863
Balanced Accuracy:0.8135411206113359, std: 0.01567835361683537
ROC-AUC: 0.9146969316661222, std: 0.010956262658794201


In [10]:
nb=NB()
nb_params={'var_smoothing': 0.0005334665971636907}
metrics=nb.evaluation(X_eval,y_eval)
metrics.out()
# nb_params=nb.hyperparameter_tuning(X_train,y_train,20)
nb.set_params(nb_params)


Accuracy: 0.7529748490467361, std: 0.01639763310586176
Precision: 0.881379100234333, std: 0.02825113072641297
F1: 0.5678335045626796, std: 0.034518124649213895
Recall: 0.4190954773869347, std: 0.03143817551681818
Balanced Accuracy:0.6918675001701179, std: 0.0191588845678086
ROC-AUC: 0.8686250983378173, std: 0.018670624831254364


In [11]:

pd.set_option('display.max_columns',None)
X_train.head(5)

,age,bmi,chol,tg,hdl,ldl,cr,bun,lipids,Age_x_BMI,HDL_x_LDL,BMI/LDL,BMI/HDL+LDL,bun_x_cr,chol/ldl,cluster_labels,Cluster 0,Cluster 1,Cluster 2,gender
0,47,25,4.40,1.24,1.27,2.65,84.1,4.50,0.479245,1175,3.3655,9.433962,6.377551,378.45,1.660377,2,11.803954,13.136126,0.915735,1
1,33,24,4.20,1.50,1.20,2.30,62.0,5.30,0.521739,792,2.7600,10.434783,6.857143,328.60,1.826087,2,11.263333,13.659371,1.380091,0
2,43,28,5.01,2.24,0.93,3.03,71.8,4.75,0.306931,1204,2.8179,9.240924,7.070707,341.05,1.653465,2,11.891569,13.804989,1.562416,1
3,60,30,5.00,3.90,1.10,2.40,47.0,4.60,0.458333,1800,2.6400,12.500000,8.571429,216.20,2.083333,2,10.243604,14.283296,3.345336,0
4,54,33,3.80,1.70,1.10,3.00,67.0,5.00,0.366667,1782,3.3000,11.000000,8.048780,335.00,1.266667,2,12.687412,13.477444,2.968891,1


In [12]:
X_train_scaled,X_test_scaled=scaling(X_train,X_test)
X_eval_scaled=pd.concat([X_train_scaled,X_test_scaled])
y_eval_scaled=y_eval
X_train_scaled.head()

,age,bmi,chol,tg,hdl,ldl,cr,bun,lipids,Age_x_BMI,HDL_x_LDL,BMI/LDL,BMI/HDL+LDL,bun_x_cr,chol/ldl,cluster_labels,Cluster 0,Cluster 1,Cluster 2,gender_1
0,-0.131754,0.094910,-0.474279,-0.363870,-0.308919,-0.280504,0.597252,-0.234099,-0.259760,-0.095656,-0.324303,-0.017079,0.124425,0.044842,-0.211829,0.333753,-0.207417,0.118558,-0.748801,1.0
1,-1.130724,-0.138833,-0.676950,-0.168124,-0.376882,-0.653040,-0.391361,0.249780,-0.127681,-0.956469,-0.433710,0.190734,0.332287,-0.098626,0.040378,0.333753,-0.393591,0.255109,-0.634671,0.0
2,-0.417174,0.796138,0.143868,0.389002,-0.639029,0.123963,0.047029,-0.082887,-0.795350,-0.030476,-0.423248,-0.057162,0.424849,-0.062795,-0.222349,0.333753,-0.177244,0.293111,-0.589858,1.0
3,0.795861,1.263623,0.133734,1.638770,-0.473974,-0.546601,-1.062365,-0.173614,-0.324759,1.309066,-0.455393,0.619561,1.075282,-0.422113,0.431902,0.333753,-0.744757,0.417934,-0.151650,0.0
4,0.367731,1.964851,-1.082292,-0.017549,-0.473974,0.092032,-0.167693,0.068325,-0.609678,1.268610,-0.336139,0.308097,0.848759,-0.080207,-0.811050,0.333753,0.096821,0.207631,-0.244173,1.0


In [13]:
mlp=MLP()
mlp_params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 6.120954197759074e-05, 'learning_rate_init': 0.006507372434299954, 'batch_size': 64, 'max_iter': 900, 'early_stopping': True, 'random_state': 42}
# params=mlp.hyperparameter_tuning(X_train_scaled,y_train,40)
# mlp.set_params(params)

mlp.set_params(mlp_params)



m=mlp.evaluation(X_eval_scaled,y_eval_scaled)
m.out()




Accuracy: 0.8299867826748443, std: 0.015051567414830007
Precision: 0.7921435593591545, std: 0.020139726911151234
F1: 0.7763794864867449, std: 0.02298501212186099
Recall: 0.7623115577889447, std: 0.038355898768364594
Balanced Accuracy:0.8175971283596233, std: 0.018357819631742957
ROC-AUC: 0.9120234792242584, std: 0.011194257455609122


In [14]:
svc=SVC()
svc_params={'C': 489.4523498582201, 'kernel': 'linear', 'gamma': 'auto', 'degree': 2, 'coef0': 0.7166123939243987, 'shrinking': True, 'tol': 0.008527769084851717, 'class_weight': 'balanced', 'probability': True, 'random_state': 42}
svc.set_params(svc_params)
# m=svc.evaluation(X_train_scaled,y_train)
# m.out()

# svc_params=svc.hyperparameter_tuning(X_train_scaled,y_train,30)


In [15]:
knn=KNN()
knn_params={'n_neighbors': 28, 'weights': 'uniform', 'metric': 'minkowski', 'p': 2, 'algorithm': 'kd_tree', 'leaf_size': 12, 'n_jobs': -1}
knn.set_params(knn_params)

m=knn.evaluation(X_eval_scaled,y_eval_scaled)
m.out()
# parameters=knn.hyperparameter_tuning(X_train_scaled,y_train,20)

Accuracy: 0.8184816241144868, std: 0.007521705174242155
Precision: 0.8212009424594859, std: 0.01620001867060146
F1: 0.7441766781240835, std: 0.01221252957602008
Recall: 0.6809045226130653, std: 0.02126049829277005
Balanced Accuracy:0.7933004718734828, std: 0.008973940601847133
ROC-AUC: 0.9060300078190402, std: 0.00677837007799398


In [16]:
ada = Ada()
ada_params={'n_estimators': 364, 'learning_rate': 0.6133741065916875}
# ada_params=ada.hyperparameter_tuning(X_eval,y_eval,20)
ada.set_params(ada_params)
m=ada.evaluation(X_eval,y_eval,5)
m.out()

Accuracy: 0.8241369276850664, std: 0.018243538692563743
Precision: 0.764328235295521, std: 0.021944611545825936
F1: 0.7771160903890741, std: 0.023410125794488694
Recall: 0.7904522613065327, std: 0.0267608582024007
Balanced Accuracy:0.8179724507093417, std: 0.019616343959428757
ROC-AUC: 0.9100594182219623, std: 0.011619251012215271


In [17]:
ensemble=[rf,xgb,ada,mlp,knn]
X_train_copy=X_train_scaled.copy()
X_test_copy=X_test_scaled.copy()
oof=pd.DataFrame()
oof_test=pd.DataFrame()
for model in ensemble:
    oof[f'OOF_{type(model).__name__}'],oof_test[f'OOF_{type(model).__name__}']=model.oof(X_train,y_train,X_test,type(model).__name__,5)

RandomForest fold 1/5 done
RandomForest fold 2/5 done
RandomForest fold 3/5 done
RandomForest fold 4/5 done
RandomForest fold 5/5 done
XGBoost fold 1/5 done
XGBoost fold 2/5 done
XGBoost fold 3/5 done
XGBoost fold 4/5 done
XGBoost fold 5/5 done
Ada fold 1/5 done
Ada fold 2/5 done
Ada fold 3/5 done
Ada fold 4/5 done
Ada fold 5/5 done
MLP fold 1/5 done
MLP fold 2/5 done
MLP fold 3/5 done
MLP fold 4/5 done
MLP fold 5/5 done
KNN fold 1/5 done
KNN fold 2/5 done
KNN fold 3/5 done
KNN fold 4/5 done
KNN fold 5/5 done


In [33]:
meta_x_model=LR()
params={'penalty': 'l2', 'l1_ratio': 0.377184927025366, 'C': 0.3926479245177274, 'class_weight': 'balanced','n_jobs':-1}

meta_x_model.set_params(params)

In [19]:
def compare(base: dict, meta: dict):
    for key in base:
        diff = meta[key] - base[key]
        symbol = "↑" if diff > 0 else ("↓" if diff < 0 else "=")
        print(f"{key}: base={base[key]:.4f} meta={meta[key]:.4f} {symbol} ({diff:+.4f})")

In [20]:
# for model in ensemble:
#     print("=================================")
#     print(f"Model: {model}")
#     if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        
#         compare(model.evaluation(X,y,5).values(),meta_x_model.evaluation(X_eval,y_eval).values())
#     else:
#         compare(model.evaluation(X_eval,y_eval,5).values(),meta_x_model.evaluation(X_eval,y_eval).values())

In [34]:
meta_x_model.train(oof,y_train)
lr_proba=meta_x_model.predict_proba(oof_test)
preds=meta_x_model.predict(oof_test)


In [22]:

def measure(y_pred: np.ndarray, y_proba: np.ndarray, y_real: np.ndarray) -> dict:
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_real, y_pred)
    metrics['balanced']=balanced_accuracy_score(y_real,y_pred)
    metrics['precision'] = precision_score(y_real, y_pred)
    metrics['f1'] = f1_score(y_real, y_pred)
    metrics['recall'] = recall_score(y_real, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_real, y_proba[:,1])
    return metrics

    

In [35]:
logistic_regression_metrics=measure(preds,lr_proba,y_test)
print(logistic_regression_metrics)
# {'accuracy': 0.815968841285297, 'balanced': 0.8144971271116122, 'precision': 0.7611607142857143, 'f1': 0.78300803673938, 'recall': 0.806146572104019, 'roc_auc': 0.9007424890016126}
# 5 folds only OOF - with FE
# {'accuracy': 0.8208373904576436, 'balanced': 0.8179277629045136, 'precision': 0.7722095671981777, 'f1': 0.7865429234338747, 'recall': 0.8014184397163121, 'roc_auc': 0.9028090899127956} 
# 4 Folds only OOF - with FE
# {'accuracy': 0.8218111002921129, 'balanced': 0.81875557747405, 'precision': 0.773972602739726, 'f1': 0.7874564459930313, 'recall': 0.8014184397163121, 'roc_auc': 0.9027386376090053}
# 4 folds only OOF - without FE
# {'accuracy': 0.8120740019474196, 'balanced': 0.8104774317786858, 'precision': 0.7566964285714286, 'f1': 0.7784156142365097, 'recall': 0.8014184397163121, 'roc_auc': 0.9007346609678579}
# 4 folds only OOF - with extra FE 
# {'accuracy': 0.8169425511197663, 'balanced': 0.8153249416811486, 'precision': 0.7628635346756152, 'f1': 0.7839080459770115, 'recall': 0.806146572104019, 'roc_auc': 0.9021750191786826}
# 4 fold only OOF - +a model
# {'accuracy': 0.8208373904576436, 'balanced': 0.8189904184866845, 'precision': 0.7685393258426966, 'f1': 0.7880184331797235, 'recall': 0.8085106382978723, 'roc_auc': 0.9033609662924866}

{'accuracy': 0.8216374269005848, 'balanced': 0.8210194281409874, 'precision': 0.7649667405764967, 'f1': 0.7903780068728522, 'recall': 0.8175355450236966, 'roc_auc': 0.9165162738143813}


In [36]:
meta_X=pd.concat([oof,oof_test])
meta_y=pd.concat([y_train,y_test])
# meta_params=meta_x_model.tune_logistic_regression(meta_X,meta_y,40)
# meta_x_model.set_params(meta_params)
metrics=meta_x_model.evaluation(meta_X,meta_y,5)
metrics.out()

Accuracy: 0.826669899681453, std: 0.012905076540369247
Precision: 0.7495940139281997, std: 0.013159516577818534
F1: 0.7879878560739876, std: 0.01682702139194338
Recall: 0.8306532663316583, std: 0.023115577889447222
Balanced Accuracy:0.8274003539472267, std: 0.014594811900901235
ROC-AUC: 0.9142762186066202, std: 0.009993503325136505


In [ ]:
for model in ensemble:
    if model.__class__.__name__ in ["XGBoost", "RandomForest","NB","Ada"]:
        model.train(X_train,y_train)
        print(model.__class__.__name__)
    else:
        model.train(X_train_copy,y_train)

RandomForest


In [ ]:
for model in ensemble:
    print(f'Model:{model}')
    if model.__class__.__name__ in ["XGBoost", "RandomForest","NB","Ada"]:
        predictions=model.predict(X_test)
        proba=model.predict_proba(X_test)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)
    else:
        predictions=model.predict(X_test_copy)
        proba=model.predict_proba(X_test_copy)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)

Model:<modeling.RandomForest.RandomForest object at 0x0000026A54F24750>
accuracy: base=0.8168 meta=0.8304 ↑ (+0.0136)
balanced: base=0.8044 meta=0.8299 ↑ (+0.0255)
precision: base=0.8031 meta=0.7756 ↓ (-0.0276)
f1: base=0.7673 meta=0.8005 ↑ (+0.0331)
recall: base=0.7346 meta=0.8270 ↑ (+0.0924)
roc_auc: base=0.9139 meta=0.9165 ↑ (+0.0026)
Model:<modeling.MLP.MLP object at 0x0000026A50E97930>
accuracy: base=0.8226 meta=0.8304 ↑ (+0.0078)
balanced: base=0.8179 meta=0.8299 ↑ (+0.0120)
precision: base=0.7804 meta=0.7756 ↓ (-0.0048)
f1: base=0.7859 meta=0.8005 ↑ (+0.0146)
recall: base=0.7915 meta=0.8270 ↑ (+0.0355)
roc_auc: base=0.9117 meta=0.9165 ↑ (+0.0047)


In [ ]:
X_train_scaled.iloc[:,-5:].corr()

,cluster_labels,Cluster 0,Cluster 1,Cluster 2,gender_1
cluster_labels,1.000000,-0.321177,0.125588,-0.829609,0.021811
Cluster 0,-0.321177,1.000000,-0.484935,0.627327,-0.053398
Cluster 1,0.125588,-0.484935,1.000000,-0.137450,0.008263
Cluster 2,-0.829609,0.627327,-0.137450,1.000000,-0.056975
gender_1,0.021811,-0.053398,0.008263,-0.056975,1.000000


In [ ]:
X_train_scaled.head()

,age,bmi,chol,tg,hdl,ldl,cr,bun,lipids,Age_x_BMI,HDL_x_LDL,BMI/LDL,BMI/HDL+LDL,bun_x_cr,chol/ldl,cluster_labels,Cluster 0,Cluster 1,Cluster 2,gender_1
0,-0.131754,0.094910,-0.474279,-0.363870,-0.308919,-0.280504,0.597252,-0.234099,-0.259760,-0.095656,-0.324303,-0.017079,0.124425,0.044842,-0.211829,0.333753,-0.207417,0.118558,-0.748801,1.0
1,-1.130724,-0.138833,-0.676950,-0.168124,-0.376882,-0.653040,-0.391361,0.249780,-0.127681,-0.956469,-0.433710,0.190734,0.332287,-0.098626,0.040378,0.333753,-0.393591,0.255109,-0.634671,0.0
2,-0.417174,0.796138,0.143868,0.389002,-0.639029,0.123963,0.047029,-0.082887,-0.795350,-0.030476,-0.423248,-0.057162,0.424849,-0.062795,-0.222349,0.333753,-0.177244,0.293111,-0.589858,1.0
3,0.795861,1.263623,0.133734,1.638770,-0.473974,-0.546601,-1.062365,-0.173614,-0.324759,1.309066,-0.455393,0.619561,1.075282,-0.422113,0.431902,0.333753,-0.744757,0.417934,-0.151650,0.0
4,0.367731,1.964851,-1.082292,-0.017549,-0.473974,0.092032,-0.167693,0.068325,-0.609678,1.268610,-0.336139,0.308097,0.848759,-0.080207,-0.811050,0.333753,0.096821,0.207631,-0.244173,1.0


In [ ]:
feature_cols=oof.columns
meta_x_model.feature_importance(feature_cols)
print(feature_cols)

            feature  coefficient
0  OOF_RandomForest     1.618959
1           OOF_MLP     1.519352
Index(['OOF_RandomForest', 'OOF_MLP'], dtype='str')
